# Compositional analysis of single cell data through scCODA package 

## Preparation of packages

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

In [ ]:
import mudata as md
import muon as mu
import mudatasets as mds
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy.external as sce
from scipy import stats
from Bio import SeqIO
from matplotlib.pyplot import rc_context
import anndata as ad
import warnings
import sccoda.util
from sccoda.util import comp_ana as mod
from sccoda.util import cell_composition_data as dat
from sccoda.util import data_visualization as viz

import sccoda.datasets as scd

warnings.filterwarnings("ignore")

In [ ]:
from matplotlib.patches import Patch

In [ ]:
sc.settings.verbosity = 4
sc.logging.print_header()
sc.settings.set_figure_params(dpi=300, facecolor='white', format = 'pdf', vector_friendly = True)

In [ ]:
umap_cmap = sns.blend_palette(['lightgrey', 'xkcd:sapphire'], as_cmap = True)

In [ ]:
figure = "Figure_7"

In [ ]:
sc.settings.figdir = './Figure_plots/'+figure

## Load and prepare dataset 

In [ ]:
adata = sc.read_h5ad('./h5ad/analysis_250528_f/Smed_L78-L47_20250523_Annotated.h5ad')
adata

##### **Subsetting only GFP and Cdh1**

In [ ]:
adata.obs.columns

In [ ]:
#View the names of the sample types
adata.obs['Sample'].unique()

In [ ]:
#Subsetting GFP and Cdh1
control_cdh1 = adata.obs['Sample'].str.contains('GFP_1|GFP_2|Cdh1_1|Cdh1_2', regex=True, case= False)
gfp_cdh1 = adata[control_cdh1, :]

In [ ]:
gfp_cdh1

**Subsetting only GFP and H2B**

In [ ]:
#Subsetting GFP and H2B
control_h2b = adata.obs['Sample'].str.contains('GFP_1|GFP_2|H2B_1|H2B_2', regex=True, case= False)
gfp_h2b = adata[control_h2b, :]

In [ ]:
gfp_h2b

**First the contrast matrix is made defining the samples and conditions of each subset** 

In [ ]:
#GFP and CDH1 Contrast Matrix
cov_df_gfp_cdh1 = pd.DataFrame({"Cond": ["GFP", "GFP", "Cdh1", "Cdh1"]},
    index=["GFP_1", "GFP_2", "Cdh1_1", "Cdh1_2"])
print(cov_df_gfp_cdh1)

In [ ]:
#GFP and H2B Contrast Matrix
cov_df_gfp_h2b = pd.DataFrame({"Cond": ["H2B", "H2B", "GFP", "GFP"]},
    index=["H2B_1", "H2B_2", "GFP_1", "GFP_2"])
print(cov_df_gfp_h2b)

In [ ]:
#Retrieve the unique values in the 'annotated_names' column of the obs dataframe within the adata object
adata.obs['annotated_names'].unique()

In [ ]:
comp_data_gfp_cdh1 = sccoda.util.cell_composition_data.from_scanpy(gfp_cdh1, 
                                        cell_type_identifier='annotated_names', 
                                        sample_identifier='Sample',
                                        covariate_df= cov_df_gfp_cdh1)

In [ ]:
comp_data_gfp_cdh1

In [ ]:
comp_data_gfp_h2b = sccoda.util.cell_composition_data.from_scanpy(gfp_h2b, 
                                        cell_type_identifier='annotated_names', 
                                        sample_identifier='Sample',
                                        covariate_df= cov_df_gfp_h2b)

In [ ]:
comp_data_gfp_h2b

**Quick visualization from <u>scCODA</u> **

In [ ]:
with plt.rc_context({'figure.figsize': (20, 5)}):
    viz.boxplots(comp_data_gfp_cdh1, 
                 feature_name="Cond", 
                 y_scale="log",
                 cmap='BuPu')# the scale in the y axis can be normal or log scale 
    viz.boxplots(comp_data_gfp_h2b, 
                 feature_name="Cond",
                 y_scale="log",
                 cmap='BuPu')
    plt.show()

## SCODA analysis

In [ ]:
model_cdh1 = mod.CompositionalAnalysis(comp_data_gfp_cdh1, formula="C(Cond, Treatment('Cdh1'))")
model_h2b = mod.CompositionalAnalysis(comp_data_gfp_h2b, formula="C(Cond, Treatment('H2B'))")

In [ ]:
sim_results_cdh1 = model_cdh1.sample_hmc()
sim_results_h2b = model_h2b.sample_hmc()

In [ ]:
sim_results_cdh1.summary()#results table can be extended with summary_extended()  

In [ ]:
sim_results_h2b.summary()

In [ ]:
print(sim_results_cdh1.credible_effects())

In [ ]:
print(sim_results_h2b.credible_effects())

**Adjusting the False Discovery Rate**

- scCODA selects credible effects based on their inclusion probability. The cutoff between credible and non-credible effects depends on the desired false discovery rate (FDR).
- For this dataset we use a FDR of 0.0.5

In [ ]:
pd.options.display.max_rows=2000 
sim_results_cdh1.set_fdr(est_fdr=0.05)
print(sim_results_cdh1.credible_effects())

In [ ]:
pd.options.display.max_rows=2000
sim_results_h2b.set_fdr(est_fdr=0.05)
print(sim_results_h2b.credible_effects())

**Secondly, for visualizing which clusters are enriched and have credible effects, we calculate the logarithm of the odd values outcome from a simple fisher test:**

In [ ]:
def FisherTest (adata, clusteringlayer, samples_name, s1, s2):
    
    cell_numbers = adata.obs[[samples_name, clusteringlayer]].groupby(samples_name)                        
    cellcounts = {}                                                                                        
    for i in adata.obs[samples_name].cat.categories:                                                       
        cellcounts[i] = cell_numbers.get_group(i)[clusteringlayer].value_counts().rename(i).sort_index()   
    counts_df = pd.DataFrame.from_dict(cellcounts)                                                         
    
    oddsdict ={}    
    pvalsdict = {}   

    for j in adata.obs[clusteringlayer].cat.categories:                                 
        cl_counts = counts_df.loc[j,[s1, s2]].to_list()                                 
        ncl_counts = (counts_df[[s1, s2]].sum() - counts_df.loc[j,[s1, s2]]).to_list()  
        oddsratio, pvalue = stats.fisher_exact([cl_counts, ncl_counts])                 
        oddsdict[j] = oddsratio                                                         
        pvalsdict[j] = pvalue                                                           
        
    counts_dict = {}   

    percs_df = counts_df[[s2, s1]] / counts_df[[s2, s1]].sum(axis = 0) * 100  
    percs_diff = percs_df[s1]-percs_df[s2]                                    
    
    fisher_df = pd.concat([pd.Series(pvalsdict),pd.Series(oddsdict), percs_diff], axis = 1, keys = ['pvals', 'odds', '% diff'])     
    
    return fisher_df, counts_df 

In [ ]:
adata.obs

In [ ]:
#1  Cdh1
clusteringlayer = 'annotated_names'
samples_name = 'Condition'
gfp = 'GFP'
cdh1 = 'Cdh1'
fisher_test_gfp_cdh1, counts_df_cdh1 = FisherTest(gfp_cdh1, clusteringlayer, samples_name, cdh1, gfp)

In [ ]:
#2  H2B
clusteringlayer = 'annotated_names'
samples_name = 'Condition'
gfp = 'GFP'
h2b = 'H2B'
fisher_test_gfp_h2b, counts_df_h2b = FisherTest(gfp_h2b, clusteringlayer, samples_name, h2b, gfp)

In [ ]:
fisher_results_gfp_cdh1 = pd.concat([fisher_test_gfp_cdh1, counts_df_cdh1], axis=1).reindex(fisher_test_gfp_cdh1.index)
fisher_results_gfp_h2b = pd.concat([fisher_test_gfp_h2b, counts_df_h2b], axis=1).reindex(fisher_test_gfp_h2b.index)

In [ ]:
fisher_results_gfp_cdh1

In [ ]:
fisher_results_gfp_h2b

In [ ]:
log2_odds_cdh1 = pd.DataFrame(np.log2(fisher_results_gfp_cdh1['odds']))
log2_odds_h2b = pd.DataFrame(np.log2(fisher_results_gfp_h2b['odds']))

visualisation of the SCoda results

In [ ]:
# get the list of enriched/depleted cell types
true_cell_types_cdh1 = sim_results_cdh1.credible_effects()[
    sim_results_cdh1.credible_effects() == True].index.get_level_values('Cell Type').unique().tolist()
# get the list of enriched/depleted cell types
true_cell_types_h2b = sim_results_h2b.credible_effects()[
    sim_results_h2b.credible_effects() == True].index.get_level_values('Cell Type').unique().tolist()

In [ ]:
li_neg_cdh1 = log2_odds_cdh1[log2_odds_cdh1['odds'] < 0].index.to_list()
li_pos_cdh1 = log2_odds_cdh1[log2_odds_cdh1['odds'] > 0].index.to_list()
ct_neg_cdh1 = [i for i in true_cell_types_cdh1 if i in li_neg_cdh1]
ct_pos_cdh1 = [i for i in true_cell_types_cdh1 if i in li_pos_cdh1]

In [ ]:
li_neg_h2b = log2_odds_h2b[log2_odds_h2b['odds'] < 0].index.to_list()
li_pos_h2b = log2_odds_h2b[log2_odds_h2b['odds'] > 0].index.to_list()
ct_neg_h2b = [i for i in true_cell_types_h2b if i in li_neg_h2b]
ct_pos_h2b = [i for i in true_cell_types_h2b if i in li_pos_h2b]

In [ ]:
with plt.rc_context({'figure.figsize': (20, 15)}):
    fig, axs = plt.subplot_mosaic([
        ['odds_g1','odds_g1', 'up_g1'],
        ['odds_g1','odds_g1', 'up_g1'],
        ['odds_g1','odds_g1', 'down_g1'],
        ['odds_g1','odds_g1', 'down_g1']],
        layout='constrained')# organization of the graphs can be adjusted accordingly
    
# Umap of clusters Up and Down in treatment G1
    sc.pl.umap(adata, color='annotated_names', legend_loc='on data', 
               legend_fontoutline = 2, 
               title= 'Credible depleted in Cdh1',
               size = 15, 
               groups = ct_neg_cdh1, 
               na_in_legend=False, na_color='#f5f5f5',
               frameon=False, show = False,
               ax = axs['up_g1'])

    sc.pl.umap(adata, color='annotated_names', legend_loc='on data', 
               legend_fontoutline = 2, 
               title= 'Credible enriched in Cdh1',
               size = 15, 
               groups = ct_pos_cdh1, 
               na_in_legend=False, na_color='#f5f5f5',
               frameon=False, show = False,
               ax = axs['down_g1'])

# Odds plot for up and down regulated clusters in G1
    value_counts_output = (log2_odds_cdh1['odds'])
    categories = adata.obs['annotated_names'].unique()
    counts = value_counts_output.values
    color_array= adata.uns['annotated_names_colors']
    color_dict = dict(zip(categories, color_array))

    axs['odds_g1']
    axs['odds_g1'].set_xlabel('Clusters')
    bars= axs['odds_g1'].barh(log2_odds_cdh1['odds'].index, 
                       (log2_odds_cdh1['odds']), 
                       edgecolor= 'black', 
                       color= [color_dict.get(category, 'gray') for category in categories]) # barh for horizontal bar plot
    axs['odds_g1'].invert_yaxis() 
    axs['odds_g1'].grid(False) # No background grid lines
    axs['odds_g1'].axvline(x=0, ymin=0, ymax=1, dashes = (2,1)) 
    axs['odds_g1'].tick_params(axis='y', labelsize=9)
    axs['odds_g1'].grid(axis='y',color='gray', linestyle='dashed', linewidth=0.5, alpha=0.5)
    axs['odds_g1'].axvline(x=0, ymin=0, ymax=1, dashes = (2,1)) 


    #plt.show()

In [ ]:
with plt.rc_context({'figure.figsize': (20, 15)}):
    fig, axs = plt.subplot_mosaic([
        ['odds_g1','odds_g1', 'up_g1'],
        ['odds_g1','odds_g1', 'up_g1'],
        ['odds_g1','odds_g1', 'down_g1'],
        ['odds_g1','odds_g1', 'down_g1']],
        layout='constrained')# organization of the graphs can be adjusted accordingly
    
# Umap of clusters Up and Down in treatment G1
    sc.pl.umap(adata, color='annotated_names', legend_loc='on data', 
               legend_fontoutline = 2, 
               title= 'Credible depleted in H2b',
               size = 15, 
               groups = ct_neg_h2b, 
               na_in_legend=False, na_color='#f5f5f5',
               frameon=False, show = False,
               ax = axs['up_g1'])

    sc.pl.umap(adata, color='annotated_names', legend_loc='on data', 
               legend_fontoutline = 2, 
               title= 'Credible enriched in H2b',
               size = 15, 
               groups = ct_pos_h2b, 
               na_in_legend=False, na_color='#f5f5f5',
               frameon=False, show = False,
               ax = axs['down_g1'])

# Odds plot for up and down regulated clusters in G1
    value_counts_output = (log2_odds_h2b['odds'])
    categories = adata.obs['annotated_names'].unique()
    counts = value_counts_output.values
    color_array= adata.uns['annotated_names_colors']
    color_dict = dict(zip(categories, color_array))

    axs['odds_g1']
    axs['odds_g1'].set_xlabel('Clusters')
    bars= axs['odds_g1'].barh(log2_odds_h2b['odds'].index, 
                       (log2_odds_h2b['odds']), 
                       edgecolor= 'black', 
                       color= [color_dict.get(category, 'gray') for category in categories]) # barh for horizontal bar plot
    axs['odds_g1'].invert_yaxis() 
    axs['odds_g1'].grid(False) # No background grid lines
    axs['odds_g1'].axvline(x=0, ymin=0, ymax=1, dashes = (2,1)) 
    axs['odds_g1'].tick_params(axis='y', labelsize=9)
    axs['odds_g1'].grid(axis='y',color='gray', linestyle='dashed', linewidth=0.5, alpha=0.5)
    axs['odds_g1'].axvline(x=0, ymin=0, ymax=1, dashes = (2,1)) 


    #plt.show()

In [ ]:
# dataframe for cdh1
neoblast_score_cdh1 = pd.DataFrame.from_dict(adata.uns['neoblast_score_annotated_names'], orient='index', columns=['neoblast_score'])
neoblast_score_cdh1.index.name = 'annotated_names'
neoblast_score_cdh1['fisher'] = log2_odds_cdh1['odds'].sort_index()

# significance column
mask = neoblast_score_cdh1.index.isin(ct_neg_cdh1)
neoblast_score_cdh1.loc[mask, 'sig'] = 'down'
mask = neoblast_score_cdh1.index.isin(ct_pos_cdh1)
neoblast_score_cdh1.loc[mask, 'sig'] = 'up'
neoblast_score_cdh1['sig'] = neoblast_score_cdh1['sig'].fillna('no')

# cluster size
neoblast_score_cdh1['cl_size'] = (fisher_results_gfp_cdh1['Cdh1'] + fisher_results_gfp_cdh1['GFP']).sort_index()

# colors
category_colors = dict(zip(adata.obs['annotated_names'].cat.categories, 
                           adata.uns['annotated_names_colors']))
neoblast_score_cdh1['colours'] = neoblast_score_cdh1.index.map(category_colors)

In [ ]:
# dataframe for h2b
neoblast_score_h2b = pd.DataFrame.from_dict(adata.uns['neoblast_score_annotated_names'], orient='index', columns=['neoblast_score'])
neoblast_score_h2b.index.name = 'annotated_names'
neoblast_score_h2b['fisher'] = log2_odds_h2b['odds'].sort_index()

# significance column
mask = neoblast_score_h2b.index.isin(ct_neg_h2b)
neoblast_score_h2b.loc[mask, 'sig'] = 'down'
mask = neoblast_score_h2b.index.isin(ct_pos_h2b)
neoblast_score_h2b.loc[mask, 'sig'] = 'up'
neoblast_score_h2b['sig'] = neoblast_score_h2b['sig'].fillna('no')

# cluster size
neoblast_score_h2b['cl_size'] = (fisher_results_gfp_h2b['H2B'] + fisher_results_gfp_h2b['GFP']).sort_index()

# colors
category_colors = dict(zip(adata.obs['annotated_names'].cat.categories, 
                           adata.uns['annotated_names_colors']))
neoblast_score_h2b['colours'] = neoblast_score_h2b.index.map(category_colors)

In [ ]:
neoblast_score_cdh1

In [ ]:
neoblast_score_h2b

In [ ]:
def bubble(neoblast_score, neoblast_plot):
    plt.figure(figsize=(12, 10))

    # edge style
    conditions = [
        neoblast_score['sig'] == 'up',
        neoblast_score['sig'] == 'down',
        neoblast_score['sig'] == 'no'
    ]
    edge_colors = np.select(conditions, ['#00008B', '#8B0000', 'grey'], default='grey')
    line_widths = np.select(conditions, [2.5, 2.5, 0.6], default=0.8)


    scatter = plt.scatter(
        x=neoblast_score['fisher'].replace(0, np.nan),
        y=neoblast_score['neoblast_score'].replace(0, np.nan),
        s=neoblast_score['cl_size'],  # bubble size
        alpha=0.8, # transparency
        color=neoblast_score['colours'],
        edgecolors = edge_colors,
        linewidths = line_widths, 
        zorder=2    
    )

    #plt.axhline(y=0.171, color='black', linestyle='--', linewidth=1, zorder=1)
    #plt.axhline(y=0.258, color='black', linestyle='--',  linewidth=1, zorder=1)
    plt.axvline(x=0, color='black', linewidth=2.5, zorder=1)
    plt.grid(False)
    plt.ylim(0, 0.4)
    plt.xlim(-2, 2)
    ax = plt.gca()

    for spine in ax.spines.values():
        spine.set_linewidth(2.5)

    #plt.ylabel('Neoblast score', fontsize=21)
    #plt.xlabel('Fisher test: log2 odd values', fontsize=21)
    plt.xticks(fontsize=28)
    plt.yticks(fontsize=28)

    plt.tight_layout()
    plt.savefig('./Figure_plots/'+ figure + '/' + neoblast_plot+ '.pdf', format='pdf')
    plt.show()


In [ ]:
# df for neoblasts / progenitors / differentiated

In [ ]:
# retrieve neoblast score for order of groups
score = pd.DataFrame(list(adata.uns['neoblast_score_annotated_names'].items()), columns=['Cell Type', 'neoblast_score'])
data = score.sort_values('neoblast_score', ascending=False).reset_index(drop=True)

In [ ]:
# list of cell types to split the violin plot in 3 catégories
li_neo = list(data.loc[data['neoblast_score'] > 0.259]['Cell Type'])
li_prog = list(data.loc[(data['neoblast_score'] < 0.259) & (data['neoblast_score'] > 0.171)]['Cell Type'])
li_diff = list(data.loc[data['neoblast_score'] < 0.171]['Cell Type'])

In [ ]:
import os
os.makedirs("./Figure_plots/Figure_7", exist_ok=True)

In [ ]:
bubble(neoblast_score_h2b, './bubble_all_h2b')

In [ ]:
bubble(neoblast_score_cdh1, 'bubble_all_cdh1')

In [ ]:
bubble(neoblast_score_h2b.loc[li_neo], 'bubble_neoblasts_h2b')

In [ ]:
bubble(neoblast_score_cdh1.loc[li_neo], 'bubble_neoblasts_cdh1')

In [ ]:
bubble(neoblast_score_h2b.loc[li_prog], 'bubble_progenitors_h2b')

In [ ]:
bubble(neoblast_score_cdh1.loc[li_prog], 'bubble_progenitors_cdh1')

In [ ]:
bubble(neoblast_score_h2b.loc[li_diff], 'bubble_differentiated_h2b')

In [ ]:
bubble(neoblast_score_cdh1.loc[li_diff], 'bubble_differentiated_cdh1')

In [ ]:
# % of cell represented by neoblasts clusters

In [ ]:
samples_name = 'Condition'
clusteringlayer = 'annotated_names'

In [ ]:
cell_numbers = adata.obs[[samples_name, clusteringlayer]].groupby(samples_name)                        
cellcounts = {}  

for i in adata.obs[samples_name].cat.categories:                                                       
    cellcounts[i] = cell_numbers.get_group(i)[clusteringlayer].value_counts().rename(i).sort_index()   
counts_df = pd.DataFrame.from_dict(cellcounts)            

In [ ]:
# number of neoblasts in the 9 clusters
counts_df.head(9)

In [ ]:
# total number of neoblasts
counts_df.head(9).sum()

In [ ]:
# number of cells per condition
counts_df.sum()

In [ ]:
percs_df = counts_df[['GFP', 'H2B', 'Cdh1']] / counts_df[['GFP', 'H2B', 'Cdh1']].sum(axis = 0) * 100  
percs_df.head(9)

In [ ]:
percs_df.head(9).sum()

In [ ]:
percs_df.sum()

In [ ]:
percs_df.head(50)

In [ ]:
percs_df.tail(50)